# 10 - Member 3 summary

**Purpose:** Collect the key numbers from the saved Member 3 runs in one place.

**Inputs:** scratch/derived/member3/*.json and reports/member3/plan.json

**Outputs:** summary table

**Status labels:** Each row keeps its own label (VERIFIED / SIMULATOR / BLOCKED / PROVISIONAL).

TEST/demo context: the demo bank in `demo_bank_app/` is SYNTHETIC.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
DERIVED.mkdir(parents=True, exist_ok=True)
# This notebook reads no credentials. If a future cell needs one, read it from an environment variable at runtime
# and never display it.

In [ ]:
def load(n):
    p = DERIVED / n
    return json.loads(p.read_text()) if p.is_file() else {}
plan, sc, ev, ma, mb, ag = (load(n) for n in ('bundle/plan.json','scheduling_comparison.json','verification_evidence.json','ml_cesnet.json','ml_purpose_triage.json','agent_eval.json'))
rows = [
 ('Plan steps READY', sum(s['state']=='READY' for s in plan.get('steps',[])), 'VERIFIED (generic loopback evidence)'),
 ('Baseline objective', sc.get('baseline',{}).get('objective'), 'VERIFIED (synthetic inputs, assumed constraints)'),
 ('QpiAI Opt authenticated solve', sc.get('qpiai_opt',{}).get('authenticated_local_solve',{}).get('status'), 'BLOCKED'),
 ('QAOA P(optimum) vs uniform', (sc.get('qaoa_phase2_wave0',{}).get('optimum_probability'), sc.get('qaoa_phase2_wave0',{}).get('uniform_probability')), 'SIMULATOR'),
 ('Compatibility cases passed', sum(r['pass'] for r in ev.get('compatibility_matrix',[])), 'VERIFIED loopback'),
 ('CESNET macro-F1 (model / majority)', (ma.get('metrics',{}).get('model',{}).get('macro_f1'), ma.get('metrics',{}).get('majority_baseline',{}).get('macro_f1')), 'VERIFIED demo; not crypto'),
 ('Purpose triage beats rules', mb.get('beats_rule_engine'), 'VERIFIED (tiny sample)'),
 ('Agent eval passed', f"{ag.get('passed')}/{ag.get('n')}", 'VERIFIED (template back-end)'),
]
import pandas as pd
pd.DataFrame(rows, columns=['item','value','status'])